In [108]:
import pandas as pd
import numpy as np
import math

# google_trends = pd.read_csv("../data/gold_google_trends_daily.csv")
data = pd.read_csv("../files/processed_data.csv")
match_context = pd.read_csv("../data/gold_match_context.csv")
# match_goals = pd.read_csv("../data/gold_match_goals.csv")
match_tickets = pd.read_csv("../data/gold_match_tickets.csv")
matches = pd.read_csv("../data/gold_match_with_league_standings.csv")
match_articles = pd.read_csv("../data/gold_belga_press_articles.csv", on_bad_lines="skip")

In [109]:
matches["stage"] = matches["stage"] != "Regular Season"

In [110]:
matches = matches[matches["is_home_match"] == True]

In [111]:
tickets_scanned_by_team = matches[["away_team_code", "away_team", "tickets_scanned", "match_id", "stage"]]
avg_tickets_scanned = matches.groupby("away_team_code")["tickets_scanned"].mean().to_frame().rename(columns={"tickets_scanned": "avg_tickets_scanned"})

In [112]:
analysing = pd.merge(tickets_scanned_by_team, avg_tickets_scanned, on="away_team_code")

In [113]:
analysing["diff"] = (analysing["tickets_scanned"] - analysing["avg_tickets_scanned"])
analysing["diff_abs"] = (analysing["tickets_scanned"] - analysing["avg_tickets_scanned"]).abs()

In [114]:
outliers = analysing[analysing["diff_abs"] > 1500]

In [115]:
weather_cols = [col for col in match_context.columns if "weather" in col.lower() or "match_id" in col.lower()]
weather_data = match_context[weather_cols]

In [116]:
exploring_weather = pd.merge(outliers, weather_data, on="match_id").drop(["weather_description", "away_team_code", "away_team"], axis=1)

exploring_weather.drop("match_id", axis=1).corr()

,tickets_scanned,stage,avg_tickets_scanned,diff,diff_abs,weather_temp_max_c,weather_temp_min_c,weather_temp_mean_c,weather_precipitation_mm,weather_rain_mm,weather_snowfall_cm,weather_windspeed_max_kmh,weather_sunshine_hours,weather_code,weather_temp_deviation,weather_score
tickets_scanned,1.000000,-0.514794,0.090154,0.906423,0.532784,0.257295,0.329051,0.294658,0.339726,0.339726,NaN,0.088722,-0.350931,0.305927,0.114008,-0.343217
stage,-0.514794,1.000000,0.103391,-0.530152,-0.224727,0.135328,-0.027542,0.057301,-0.204700,-0.204700,NaN,-0.208923,0.534985,-0.135975,-0.220635,0.142276
avg_tickets_scanned,0.090154,0.103391,1.000000,-0.338933,-0.003556,0.586154,0.410752,0.512729,-0.156206,-0.156206,NaN,-0.331474,0.374686,-0.195149,-0.173961,0.148125
diff,0.906423,-0.530152,-0.338933,1.000000,0.504807,-0.005531,0.136642,0.060904,0.387171,0.387171,NaN,0.224390,-0.490413,0.371759,0.181475,-0.387042
diff_abs,0.532784,-0.224727,-0.003556,0.504807,1.000000,0.013894,0.087917,0.038086,0.240265,0.240265,NaN,0.025824,-0.182094,0.381113,0.074307,-0.317702
weather_temp_max_c,0.257295,0.135328,0.586154,-0.005531,0.013894,1.000000,0.901535,0.981236,0.174905,0.174905,NaN,-0.318597,0.396690,-0.056260,0.393160,0.034095
weather_temp_min_c,0.329051,-0.027542,0.410752,0.136642,0.087917,0.901535,1.000000,0.960461,0.291008,0.291008,NaN,-0.186797,0.167270,0.194525,0.496405,-0.140084
weather_temp_mean_c,0.294658,0.057301,0.512729,0.060904,0.038086,0.981236,0.960461,1.000000,0.213274,0.213274,NaN,-0.258012,0.301514,0.046867,0.461117,-0.024063
weather_precipitation_mm,0.339726,-0.204700,-0.156206,0.387171,0.240265,0.174905,0.291008,0.213274,1.000000,1.000000,NaN,0.450901,-0.286264,0.592172,0.192299,-0.875218
weather_rain_mm,0.339726,-0.204700,-0.156206,0.387171,0.240265,0.174905,0.291008,0.213274,1.000000,1.000000,NaN,0.450901,-0.286264,0.592172,0.192299,-0.875218


In [118]:
seasonpass_holders = match_tickets[["seasonpass_holders", "match_id"]]


pd.merge(exploring_weather, seasonpass_holders, on="match_id").drop("match_id", axis=1).corr()

,tickets_scanned,stage,avg_tickets_scanned,diff,diff_abs,weather_temp_max_c,weather_temp_min_c,weather_temp_mean_c,weather_precipitation_mm,weather_rain_mm,weather_snowfall_cm,weather_windspeed_max_kmh,weather_sunshine_hours,weather_code,weather_temp_deviation,weather_score,seasonpass_holders
tickets_scanned,1.000000,-0.514794,0.090154,0.906423,0.532784,0.257295,0.329051,0.294658,0.339726,0.339726,NaN,0.088722,-0.350931,0.305927,0.114008,-0.343217,0.176620
stage,-0.514794,1.000000,0.103391,-0.530152,-0.224727,0.135328,-0.027542,0.057301,-0.204700,-0.204700,NaN,-0.208923,0.534985,-0.135975,-0.220635,0.142276,0.488370
avg_tickets_scanned,0.090154,0.103391,1.000000,-0.338933,-0.003556,0.586154,0.410752,0.512729,-0.156206,-0.156206,NaN,-0.331474,0.374686,-0.195149,-0.173961,0.148125,-0.085406
diff,0.906423,-0.530152,-0.338933,1.000000,0.504807,-0.005531,0.136642,0.060904,0.387171,0.387171,NaN,0.224390,-0.490413,0.371759,0.181475,-0.387042,0.203066
diff_abs,0.532784,-0.224727,-0.003556,0.504807,1.000000,0.013894,0.087917,0.038086,0.240265,0.240265,NaN,0.025824,-0.182094,0.381113,0.074307,-0.317702,0.223195
weather_temp_max_c,0.257295,0.135328,0.586154,-0.005531,0.013894,1.000000,0.901535,0.981236,0.174905,0.174905,NaN,-0.318597,0.396690,-0.056260,0.393160,0.034095,0.196359
weather_temp_min_c,0.329051,-0.027542,0.410752,0.136642,0.087917,0.901535,1.000000,0.960461,0.291008,0.291008,NaN,-0.186797,0.167270,0.194525,0.496405,-0.140084,0.138417
weather_temp_mean_c,0.294658,0.057301,0.512729,0.060904,0.038086,0.981236,0.960461,1.000000,0.213274,0.213274,NaN,-0.258012,0.301514,0.046867,0.461117,-0.024063,0.167759
weather_precipitation_mm,0.339726,-0.204700,-0.156206,0.387171,0.240265,0.174905,0.291008,0.213274,1.000000,1.000000,NaN,0.450901,-0.286264,0.592172,0.192299,-0.875218,0.081944
weather_rain_mm,0.339726,-0.204700,-0.156206,0.387171,0.240265,0.174905,0.291008,0.213274,1.000000,1.000000,NaN,0.450901,-0.286264,0.592172,0.192299,-0.875218,0.081944


In [123]:
exploring_weather[exploring_weather["stage"] == 0]["tickets_scanned"].sum() / len(
    exploring_weather[exploring_weather["stage"] == 0]
)

exploring_weather[exploring_weather["stage"] == 1]["tickets_scanned"].sum() / len(exploring_weather[exploring_weather["stage"] == 1])

np.float64(7798.823529411765)

In [ ]:
exploring_weather.groupby("weather_temp_min_c")[]